In [2]:
import pandas as pd

crime = pd.read_csv('../data/processed/crime_with_neighborhoods.csv')
transit = pd.read_csv('../data/processed/transit_bu_scores.csv')
rent = pd.read_csv('../data/processed/rent_scores.csv')
amenities = pd.read_csv('../data/processed/amenity_scores.csv')

print(crime.shape)
print(transit.shape)
print(rent.shape)
print(amenities.shape)

(236448, 6)
(26, 3)
(20, 2)
(26, 2)


In [3]:
crime_counts = crime.groupby('neighborhood')['OFFENSE_DESCRIPTION'].count().reset_index()
crime_counts.columns = ['neighborhood', 'crime_count']
print(crime_counts.sort_values('crime_count', ascending=False).head())

    neighborhood  crime_count
7     Dorchester        54640
20       Roxbury        28153
23     South End        15430
21  South Boston        14064
8       Downtown        13603


In [4]:
# Start with rent as base (only 20 neighborhoods we care about)
master = rent.copy()
master.columns = ['neighborhood', 'avg_rent']

# Merge crime counts
master = master.merge(crime_counts, on='neighborhood', how='left')

# Merge transit and BU distance
master = master.merge(transit, on='neighborhood', how='left')

# Merge amenities
master = master.merge(amenities, on='neighborhood', how='left')

print(master.shape)
master.head()

KeyError: 'neighborhood'

In [5]:
print(transit.columns.tolist())
print(amenities.columns.tolist())

['name', 'transit_score', 'bu_distance_miles']
['neighborhood', 'amenity_count']


In [6]:
transit = transit.rename(columns={'name': 'neighborhood'})
print(transit.columns.tolist())

['neighborhood', 'transit_score', 'bu_distance_miles']


In [7]:
# Start with rent as base (only 20 neighborhoods we care about)
master = rent.copy()
master.columns = ['neighborhood', 'avg_rent']

# Merge crime counts
master = master.merge(crime_counts, on='neighborhood', how='left')

# Merge transit and BU distance
master = master.merge(transit, on='neighborhood', how='left')

# Merge amenities
master = master.merge(amenities, on='neighborhood', how='left')

print(master.shape)
master.head()

(20, 6)


,neighborhood,avg_rent,crime_count,transit_score,bu_distance_miles,amenity_count
0,Allston,3653.71,5407,35,1.31,121
1,Back Bay,3446.92,9813,57,1.27,125
2,Beacon Hill,3433.75,2604,41,1.95,25
3,Brighton,3031.61,11415,27,2.54,74
4,Charlestown,3326.28,3948,23,2.91,16


In [8]:
print(master.shape)
master.isnull().sum()

(20, 6)


neighborhood         0
avg_rent             0
crime_count          0
transit_score        0
bu_distance_miles    0
amenity_count        0
dtype: int64

In [9]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Normalize all metrics
master['rent_norm'] = scaler.fit_transform(master[['avg_rent']])
master['crime_norm'] = scaler.fit_transform(master[['crime_count']])
master['transit_norm'] = scaler.fit_transform(master[['transit_score']])
master['bu_norm'] = scaler.fit_transform(master[['bu_distance_miles']])
master['amenity_norm'] = scaler.fit_transform(master[['amenity_count']])

# Flip the ones where lower is better
master['rent_norm'] = 1 - master['rent_norm']
master['crime_norm'] = 1 - master['crime_norm']
master['bu_norm'] = 1 - master['bu_norm']

In [10]:
master[['neighborhood', 'rent_norm', 'crime_norm', 'transit_norm', 'bu_norm', 'amenity_norm']].head()

,neighborhood,rent_norm,crime_norm,transit_norm,bu_norm,amenity_norm
0,Allston,0.667308,0.926704,0.607143,0.886139,0.965517
1,Back Bay,0.737572,0.843771,1.000000,0.892739,1.000000
2,Beacon Hill,0.742047,0.979464,0.714286,0.780528,0.137931
3,Brighton,0.878687,0.813616,0.464286,0.683168,0.560345
4,Charlestown,0.778563,0.954166,0.392857,0.622112,0.060345


In [11]:
master['final_score'] = (
    master['bu_norm'] * 0.40 +
    master['crime_norm'] * 0.30 +
    master['transit_norm'] * 0.20 +
    master['rent_norm'] * 0.10 +
    master['amenity_norm'] * 0.10
)

master[['neighborhood', 'final_score']].sort_values('final_score', ascending=False)

,neighborhood,final_score
8,Fenway,1.002891
1,Back Bay,0.983984
0,Allston,0.917178
5,Chinatown,0.839303
2,Beacon Hill,0.836905
17,South End,0.813444
11,Mission Hill,0.797340
16,South Boston Waterfront,0.768759
12,North End,0.756910
18,West End,0.756802


In [12]:
master.to_csv('../data/processed/master_scores.csv', index=False)
print("Saved!")

Saved!
